# Notebook 2 — DDR1 crystal-pose redocking (5SB1 / 1JI)

Pipeline:
1. Clean 5SB1 (chain A, alternate locations resolved).
2. Prepare receptor with Meeko.
3. Extract crystallographic 1JI and prepare the ligand with Meeko while preserving heavy-atom coordinates.
4. Derive the docking box from the crystallographic 1JI centroid and save it to `redocking_box.json`.
5. Redock with seeds 7, 42, and 2026.
6. Validate coordinate-preserved SDF-to-PDBQT atom correspondence (including ambiguity checks and an audit CSV) and calculate symmetry-corrected direct heavy-atom RMSD **without superposition**.
7. Compare crystal and best-redocked-pose interactions with PLIP.

RMSD is only considered valid if the input ligand coordinate mapping passes QC and Vina output atom order matches the prepared input ligand. The chemical graph is sourced from the sanitized reference SDF because PDBQT does not encode the full bond graph. The symmetry correction uses graph automorphisms of that reference ligand; no coordinate fitting is performed. Review the atom-mapping audit CSV and source ligand chemistry before treating RMSD as definitive.


In [ ]:
#Installing the required packages

%pip install meeko==0.8.0
!apt-get update -qq
!apt-get install -y autodock-vina
%pip install rdkit
%pip -q install biopython
%pip install gemmi
%pip install prody
%pip install -q plip
plip --version
python -m pip show plip
import Bio
print("Biopython version:", Bio.__version__)


W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu noble InRelease' does not seem to provide it (sources.list entry misspelt?)
Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
autodock-vina is already the newest version (1.2.5-2build1).
0 upgraded, 0 newly installed, 0 to remove and 24 not upgraded.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 99.1/99.1 kB 2.1 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.6/12.6 MB 59.3 MB/s eta 0:00:00:00:01:01
Biopython version: 1.88


In [2]:
# CELL 1 — Config, paths, environment
import sys, json, shutil, subprocess
from pathlib import Path

PROJECT_NAME = "DDR1_Workflow"
try:
    from google.colab import drive
    if not Path("/content/drive/MyDrive").exists():
        drive.mount("/content/drive")
except ImportError:
    pass
_search = [Path("/content/drive/MyDrive") / PROJECT_NAME, Path("/content"),
           Path.home(), Path.cwd(), *Path.cwd().parents]
for _c in _search:
    if (_c / "ddr1_config.py").exists():
        sys.path.insert(0, str(_c)); break
else:
    raise FileNotFoundError("ddr1_config.py not found. Run 0-DDR1-CONFIG first.")
sys.modules.pop("ddr1_config", None)

from ddr1_config import setup_paths, sha256_file, write_ligand_pdb, direct_rmsd

PATHS = setup_paths(project_name=PROJECT_NAME)
PROJECT = PATHS["REDOCKING"]
RAW_STRUCTURE_DIR = PATHS["REDOCKING_RAW_STRUCTURE"]
RECEPTOR_DIR = PATHS["REDOCKING_RECEPTOR"]
LIGAND_DIR = PATHS["REDOCKING_LIGAND"]
DOCKING_DIR = PATHS["REDOCKING_DOCKING"]
ANALYSIS_DIR = PATHS["REDOCKING_ANALYSIS"]
REPORTS_DIR = PATHS["REDOCKING_REPORTS"]

RAW_PDB = RAW_STRUCTURE_DIR / "5SB1.pdb"
RECEPTOR_PDB = RECEPTOR_DIR / "5SB1_receptor.pdb"
RECEPTOR_PDBQT = RECEPTOR_DIR / "5SB1_receptor.pdbqt"
CRYSTAL_LIGAND_PDB = LIGAND_DIR / "reference_1JI_crystal.pdb"
CRYSTAL_LIGAND_SDF = LIGAND_DIR / "reference_1JI_crystal.sdf"
CRYSTAL_LIGAND_EXPLICIT_H_SDF = LIGAND_DIR / "reference_1JI_crystal_explicitH.sdf"
CRYSTAL_LIGAND_PDBQT = LIGAND_DIR / "1JI_docking.pdbqt"
REDOCKING_1JI_DIR = DOCKING_DIR / "1JI_redocking"
REDOCKING_1JI_DIR.mkdir(parents=True, exist_ok=True)
BOX_JSON = REPORTS_DIR / "redocking_box.json"

SEEDS = [7, 42, 2026]
BOX_SIZE = (22.0, 22.0, 22.0)
EXHAUSTIVENESS = 32
NUM_MODES = 20
ENERGY_RANGE = 4
CPU = 2

MEEKO_RECEPTOR = Path(shutil.which("mk_prepare_receptor.py") or "/usr/local/bin/mk_prepare_receptor.py")
MEEKO_LIGAND = Path(shutil.which("mk_prepare_ligand.py") or "/usr/local/bin/mk_prepare_ligand.py")
VINA = Path(shutil.which("vina") or "/usr/local/bin/vina")

if not RAW_PDB.exists():
    raise FileNotFoundError(f"Upload 5SB1.pdb into {RAW_STRUCTURE_DIR} first")
for exe, lab in [(MEEKO_RECEPTOR, "mk_prepare_receptor"), (MEEKO_LIGAND, "mk_prepare_ligand"), (VINA, "vina")]:
    if not exe.exists():
        raise FileNotFoundError(f"Missing {lab}: {exe}")
print("5SB1:", RAW_PDB)
print("Meeko:", MEEKO_RECEPTOR, MEEKO_LIGAND)
print("Vina :", VINA)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Environment : COLAB
Project Root: /content/drive/MyDrive/DDR1_Workflow
Directories : 30
5SB1: /content/drive/MyDrive/DDR1_Workflow/ddr1_redocking/raw_structure/5SB1.pdb
Meeko: /usr/local/bin/mk_prepare_receptor.py /usr/local/bin/mk_prepare_ligand.py
Vina : /usr/bin/vina


In [3]:
# CELL 2 — Receptor cleaning (chain A, altloc resolved)
from Bio.PDB import PDBParser, PDBIO, Select

ALTLOC_SELECTIONS = {("A", 676, "MET"): "A", ("A", 831, "LEU"): "A",
                     ("A", 841, "GLN"): "B", ("A", 878, "MET"): "A"}
STANDARD_RESIDUES = {"ALA","ARG","ASN","ASP","CYS","GLN","GLU","GLY",
                     "HIS","ILE","LEU","LYS","MET","PHE","PRO","SER",
                     "THR","TRP","TYR","VAL","MSE"}

class ReceptorSelect(Select):
    def accept_chain(self, chain):
        return chain.id == "A"
    def accept_residue(self, residue):
        return residue.resname.strip() in STANDARD_RESIDUES
    def accept_atom(self, atom):
        res = atom.get_parent(); ch = res.get_parent()
        key = (ch.id, res.id[1], res.resname.strip())
        if atom.is_disordered():
            sel = ALTLOC_SELECTIONS.get(key)
            if sel is not None:
                if atom.get_altloc() == sel:
                    atom.set_altloc(" "); return True
                return False
            chosen = atom.selected_child
            if chosen is None or atom is not chosen:
                return False
            atom.set_altloc(" "); return True
        atom.set_altloc(" ")
        return True

parser = PDBParser(QUIET=True)
structure = parser.get_structure("5SB1", str(RAW_PDB))
model = list(structure.get_models())[0]
if "A" not in [c.id for c in model]:
    raise RuntimeError("Chain A missing from 5SB1")
io = PDBIO(); io.set_structure(structure)
io.save(str(RECEPTOR_PDB), ReceptorSelect())

atom_count = hetatm = altloc = 0
for line in RECEPTOR_PDB.read_text().splitlines():
    if line.startswith("ATOM"):
        atom_count += 1
        if len(line) > 16 and line[16].strip():
            altloc += 1
    elif line.startswith("HETATM"):
        hetatm += 1
print(f"Receptor PDB: ATOM={atom_count} HETATM={hetatm} altloc={altloc}")
if hetatm or altloc:
    raise RuntimeError("Receptor cleaning failed QC")

Receptor PDB: ATOM=2283 HETATM=0 altloc=0


In [4]:
# CELL 3 — Receptor PDBQT with Meeko
if RECEPTOR_PDBQT.exists():
    RECEPTOR_PDBQT.unlink()
output_base = RECEPTOR_DIR / "5SB1_receptor"
cmd = [str(MEEKO_RECEPTOR), "-i", str(RECEPTOR_PDB), "-o", str(output_base), "--write_pdbqt"]
res = subprocess.run(cmd, capture_output=True, text=True)
if res.returncode != 0:
    raise RuntimeError(f"Meeko receptor failed:\n{res.stdout}\n{res.stderr}")
if not RECEPTOR_PDBQT.exists():
    RECEPTOR_PDBQT = output_base.with_suffix(".pdbqt")
if not RECEPTOR_PDBQT.exists():
    raise FileNotFoundError("Meeko receptor output missing")
pdbqt_atoms = sum(1 for L in RECEPTOR_PDBQT.read_text(errors="replace").splitlines()
                  if L.startswith(("ATOM", "HETATM")))
print(f"Receptor PDBQT: {RECEPTOR_PDBQT}  ({pdbqt_atoms} atoms)")
print(f"Receptor SHA256: {sha256_file(RECEPTOR_PDBQT)}")

Receptor PDBQT: /content/drive/MyDrive/DDR1_Workflow/ddr1_redocking/receptor/5SB1_receptor.pdbqt  (2791 atoms)
Receptor SHA256: f52b93259515a7802d605245c78127c886b24163ab040539cd873fb7ed588a74


In [5]:
# CELL 4 — Extract 1JI, derive BOX from centroid, save box.json
import numpy as np
from rdkit import Chem

structure = parser.get_structure("5SB1", str(RAW_PDB))
model = list(structure.get_models())[0]
target = None
for chain in model:
    for residue in chain:
        if residue.resname.strip() == "1JI" and chain.id == "A" and residue.id[1] == 1001:
            target = (chain, residue)
if target is None:
    raise RuntimeError("1JI not found")
chain, ligand_res = target
lig_atoms = list(ligand_res.get_atoms())

class LigandSelect(Select):
    def accept_chain(self, c): return c.id == "A"
    def accept_residue(self, r): return r is ligand_res
    def accept_atom(self, a): return True

if CRYSTAL_LIGAND_PDB.exists():
    CRYSTAL_LIGAND_PDB.unlink()
io = PDBIO(); io.set_structure(structure)
io.save(str(CRYSTAL_LIGAND_PDB), LigandSelect())

coords = np.array([list(a.coord) for a in lig_atoms])
centroid = coords.mean(axis=0)
print(f"1JI centroid (box center): {centroid.tolist()}")

BOX = {"center_x": float(centroid[0]),
       "center_y": float(centroid[1]),
       "center_z": float(centroid[2]),
       "size_x": BOX_SIZE[0], "size_y": BOX_SIZE[1], "size_z": BOX_SIZE[2],
       "derivation": "centroid of crystallographic 1JI in 5SB1 chain A"}
BOX_JSON.write_text(json.dumps(BOX, indent=2))
print(f"Box written: {BOX_JSON}")

# PDB -> SDF preserving coordinates
mol = Chem.MolFromPDBFile(str(CRYSTAL_LIGAND_PDB), sanitize=False, removeHs=False)
if mol is None:
    raise RuntimeError("RDKit could not read 1JI PDB")
conf = mol.GetConformer()
orig = np.array([list(conf.GetAtomPosition(i)) for i in range(mol.GetNumAtoms())])
Chem.SanitizeMol(mol)
new = np.array([list(mol.GetConformer().GetAtomPosition(i)) for i in range(mol.GetNumAtoms())])
if float(np.linalg.norm(new - orig, axis=1).max()) > 1e-6:
    raise RuntimeError("Sanitization moved coordinates")
w = Chem.SDWriter(str(CRYSTAL_LIGAND_SDF)); w.write(mol); w.close()
print(f"1JI SDF: {CRYSTAL_LIGAND_SDF}")

1JI centroid (box center): [22.764719009399414, -2.076000213623047, -1.1666250228881836]
Box written: /content/drive/MyDrive/DDR1_Workflow/ddr1_redocking/reports/redocking_box.json
1JI SDF: /content/drive/MyDrive/DDR1_Workflow/ddr1_redocking/ligand/reference_1JI_crystal.sdf


In [6]:
# CELL 5 — Add H, prepare ligand PDBQT
mol = next(m for m in Chem.SDMolSupplier(str(CRYSTAL_LIGAND_SDF), removeHs=False) if m is not None)
heavy_idx = [a.GetIdx() for a in mol.GetAtoms() if a.GetAtomicNum() > 1]
before = np.array([list(mol.GetConformer().GetAtomPosition(i)) for i in heavy_idx])
mol_h = Chem.AddHs(mol, addCoords=True)
after = np.array([list(mol_h.GetConformer().GetAtomPosition(i)) for i in heavy_idx])
max_d = float(np.linalg.norm(after - before, axis=1).max())
if max_d > 1e-6:
    raise RuntimeError(f"AddHs moved heavy atoms by {max_d:.6f} A")

if CRYSTAL_LIGAND_EXPLICIT_H_SDF.exists():
    CRYSTAL_LIGAND_EXPLICIT_H_SDF.unlink()
w = Chem.SDWriter(str(CRYSTAL_LIGAND_EXPLICIT_H_SDF)); w.write(mol_h); w.close()

if CRYSTAL_LIGAND_PDBQT.exists():
    CRYSTAL_LIGAND_PDBQT.unlink()
cmd = [str(MEEKO_LIGAND), "-i", str(CRYSTAL_LIGAND_EXPLICIT_H_SDF),
       "-o", str(CRYSTAL_LIGAND_PDBQT)]
res = subprocess.run(cmd, capture_output=True, text=True)
if res.returncode != 0 or not CRYSTAL_LIGAND_PDBQT.exists():
    raise RuntimeError(f"Meeko ligand failed:\n{res.stdout}\n{res.stderr}")
print(f"Ligand PDBQT: {CRYSTAL_LIGAND_PDBQT}  SHA256: {sha256_file(CRYSTAL_LIGAND_PDBQT)}")

Ligand PDBQT: /content/drive/MyDrive/DDR1_Workflow/ddr1_redocking/ligand/1JI_docking.pdbqt  SHA256: 65f147184cdaa0c6ef3d51049d998040ef2f17b127e8e642487cf08e5ee97eae


In [7]:
# CELL 6 — Redock at three seeds
import pandas as pd
run_rows = []
for seed in SEEDS:
    out_pdbqt = REDOCKING_1JI_DIR / f"1JI_redocked_seed_{seed}.pdbqt"
    log_file = REDOCKING_1JI_DIR / f"1JI_redocked_seed_{seed}.log"
    cmd = [str(VINA),
           "--receptor", str(RECEPTOR_PDBQT),
           "--ligand", str(CRYSTAL_LIGAND_PDBQT),
           "--center_x", str(BOX["center_x"]),
           "--center_y", str(BOX["center_y"]),
           "--center_z", str(BOX["center_z"]),
           "--size_x", str(BOX["size_x"]),
           "--size_y", str(BOX["size_y"]),
           "--size_z", str(BOX["size_z"]),
           "--exhaustiveness", str(EXHAUSTIVENESS),
           "--num_modes", str(NUM_MODES),
           "--energy_range", str(ENERGY_RANGE),
           "--cpu", str(CPU),
           "--seed", str(seed),
           "--out", str(out_pdbqt)]
    r = subprocess.run(cmd, capture_output=True, text=True)
    log_file.write_text(" ".join(cmd) + "\n\n" + (r.stdout or "") + "\n" + (r.stderr or ""))
    ok = r.returncode == 0 and out_pdbqt.exists()
    run_rows.append({"seed": seed, "rc": r.returncode, "output": str(out_pdbqt),
                     "status": "SUCCESS" if ok else "FAILED"})
    print(f"seed {seed:>5}  rc={r.returncode}  {'OK' if ok else 'FAILED'}")
summary = pd.DataFrame(run_rows)
summary.to_csv(REDOCKING_1JI_DIR / "redocking_run_summary.csv", index=False)
if not (summary["status"] == "SUCCESS").all():
    raise RuntimeError("At least one redocking run failed")

seed     7  rc=0  OK
seed    42  rc=0  OK
seed  2026  rc=0  OK


In [9]:
# CELL 7 — Atom-mapped, symmetry-corrected direct heavy-atom RMSD
# No translation or rotation/superposition is applied.
#
# Strategy:
# 1. Parse PDBQT using AutoDock atom types (not the element column; Cl can be misread there).
# 2. Establish the SDF <-> prepared PDBQT mapping from preserved coordinates, with
#    element constraints, nearest-neighbor uniqueness checks, and strict coordinate QC.
#    Save an atom-mapping audit CSV with reference-graph neighborhood annotations.
# 3. Require each Vina output to preserve heavy-atom count and ordered element/type records.
#    Atom names are retained for audit but are not assumed to be unique.
# 4. Reorder each pose into reference-SDF atom order.
# 5. Enumerate graph-preserving automorphisms of the reference ligand and take the minimum
#    direct RMSD across valid mappings. No superposition is used.

from pathlib import Path
from collections import Counter
import numpy as np
import pandas as pd
from scipy.optimize import linear_sum_assignment
from rdkit import Chem

AUTODOCK_TO_ELEMENT = {
    "C": "C", "A": "C",
    "N": "N", "NA": "N", "NS": "N",
    "O": "O", "OA": "O", "OS": "O",
    "S": "S", "SA": "S", "P": "P",
    "F": "F", "CL": "CL", "BR": "BR", "I": "I",
    "H": "H", "HD": "H", "HS": "H",
    "D": "H",
}


def parse_pdbqt_heavy_atoms(path):
    """Parse the first PDBQT model into ordered heavy-atom records."""
    path = Path(path)
    if not path.is_file():
        raise FileNotFoundError(f"PDBQT file not found: {path}")

    records = []
    model_started = False
    saw_model_record = False

    for line in path.read_text(errors="replace").splitlines():
        if line.startswith("MODEL"):
            if saw_model_record:
                break
            saw_model_record = True
            model_started = True
            continue
        if line.startswith("ENDMDL") and model_started:
            break
        if not line.startswith(("ATOM  ", "HETATM")):
            continue

        # A PDBQT without MODEL records is treated as one structure.
        if saw_model_record and not model_started:
            continue

        try:
            xyz = np.array([
                float(line[30:38]),
                float(line[38:46]),
                float(line[46:54]),
            ], dtype=float)
        except ValueError as exc:
            raise ValueError(f"Invalid coordinates in {path}: {line}") from exc

        atom_type = line[77:79].strip().upper() if len(line) >= 79 else ""
        element = AUTODOCK_TO_ELEMENT.get(atom_type)
        if element is None:
            raise RuntimeError(
                f"Unrecognized AutoDock atom type {atom_type!r} in {path.name}: {line}"
            )
        if element == "H":
            continue

        records.append({
            "element": element,
            "xyz": xyz,
            "atom_name": line[12:16].strip(),
            "atom_type": atom_type,
            "line": line,
        })

    if not records:
        raise RuntimeError(f"No heavy atoms found in {path}")
    return records


def get_heavy_reference(sdf_path):
    supplier = Chem.SDMolSupplier(str(sdf_path), removeHs=False, sanitize=True)
    mol = next((m for m in supplier if m is not None), None)
    if mol is None:
        raise RuntimeError(f"Could not read valid reference SDF: {sdf_path}")
    if mol.GetNumConformers() == 0:
        raise RuntimeError("Reference ligand SDF has no 3D conformer")

    heavy = Chem.RemoveHs(mol)
    if heavy.GetNumConformers() == 0:
        raise RuntimeError("Heavy-atom reference has no coordinates")
    conf = heavy.GetConformer()
    coords = np.asarray([
        list(conf.GetAtomPosition(i)) for i in range(heavy.GetNumAtoms())
    ], dtype=float)
    elements = [a.GetSymbol().upper() for a in heavy.GetAtoms()]
    return mol, heavy, coords, elements


def coordinate_mapping_sdf_to_pdbqt(ref_coords, ref_elements, pdbqt_records,
                                     tolerance_angstrom=0.01,
                                     uniqueness_margin_angstrom=0.05):
    """Recover PDBQT-order -> reference-SDF-index mapping with strict QC.

    PDBQT has no complete bond graph, so this function does NOT infer chemistry
    from PDBQT. It establishes atom identity only from preserved coordinates and
    element identity; the chemical graph remains that of the reference SDF.
    Ambiguous coordinate assignments fail rather than being silently accepted.
    """
    p_coords = np.asarray([r["xyz"] for r in pdbqt_records], dtype=float)
    p_elements = [r["element"] for r in pdbqt_records]

    if len(ref_coords) != len(p_coords):
        raise RuntimeError(
            f"Input ligand heavy-atom count mismatch: SDF={len(ref_coords)}, PDBQT={len(p_coords)}"
        )
    if Counter(ref_elements) != Counter(p_elements):
        raise RuntimeError(
            "Input ligand element composition mismatch.\n"
            f"SDF:   {Counter(ref_elements)}\nPDBQT: {Counter(p_elements)}"
        )

    # PDBQT atom names are labels, not guaranteed unique identifiers. Some
    # converters emit generic names such as C, N, and O repeatedly. Record them
    # for audit, but do not reject duplicates or use names as unique atom IDs.
    names = [r["atom_name"] for r in pdbqt_records]
    if any(not n for n in names):
        raise RuntimeError("Prepared PDBQT contains a blank heavy-atom name")
    duplicate_names = sorted(n for n in set(names) if names.count(n) > 1)
    if duplicate_names:
        print("Note: repeated generic PDBQT atom labels (not used as unique IDs):",
              duplicate_names)

    input_to_ref = np.full(len(p_coords), -1, dtype=int)
    matched_distances = np.full(len(p_coords), np.inf, dtype=float)
    uniqueness_gaps = np.full(len(p_coords), np.inf, dtype=float)

    for element in sorted(set(ref_elements)):
        ref_idx = [i for i, e in enumerate(ref_elements) if e == element]
        pdb_idx = [i for i, e in enumerate(p_elements) if e == element]
        dmat = np.linalg.norm(
            p_coords[pdb_idx, None, :] - ref_coords[None, ref_idx, :], axis=2
        )
        p_local, r_local = linear_sum_assignment(dmat)
        for pp, rr in zip(p_local, r_local):
            p_i, r_i = pdb_idx[pp], ref_idx[rr]
            input_to_ref[p_i] = r_i
            matched_distances[p_i] = dmat[pp, rr]
            # Record nearest-neighbour gap for diagnostics only. Near-ties can
            # occur for symmetry-related atoms and are not automatically failures.
            sorted_d = np.sort(dmat[pp])
            if len(sorted_d) > 1:
                uniqueness_gaps[p_i] = float(sorted_d[1] - sorted_d[0])

    if np.any(input_to_ref < 0) or len(set(input_to_ref.tolist())) != len(ref_coords):
        raise RuntimeError("Input SDF/PDBQT atom mapping is not one-to-one")

    # Do not require each atom's independent nearest neighbour to be unique.
    # The Hungarian assignment is globally one-to-one; strict coordinate QC below
    # verifies that the assigned mapping reproduces the original reference pose.

    max_dev = float(matched_distances.max())
    rms_dev = float(np.sqrt(np.mean(matched_distances ** 2)))
    print(f"Input SDF-to-PDBQT mapping RMSD: {rms_dev:.6f} A")
    print(f"Input SDF-to-PDBQT maximum deviation: {max_dev:.6f} A")
    print(f"Minimum same-element nearest-neighbor gap: {np.min(uniqueness_gaps):.6f} A")
    if max_dev > tolerance_angstrom:
        raise RuntimeError(
            f"Input coordinate-preservation QC failed (max={max_dev:.4f} A; "
            f"threshold={tolerance_angstrom:.3f} A). Do not calculate/report redocking RMSD "
            "until the source ligand and atom mapping are reconciled."
        )
    return input_to_ref, matched_distances, uniqueness_gaps


def direct_symmetry_corrected_rmsd(ref_mol_heavy, ref_coords, pose_coords_ref_order,
                                   max_automorphisms=1000000):
    """Minimum graph-valid heavy-atom RMSD across reference self-mappings; no fitting."""
    n = ref_mol_heavy.GetNumAtoms()
    if ref_coords.shape != (n, 3) or pose_coords_ref_order.shape != (n, 3):
        raise ValueError("Coordinate arrays do not match the reference heavy-atom graph")

    # A self-substructure match gives graph-preserving atom permutations. Since the query
    # and target are the same sanitized heavy-atom molecule, element, bond order, aromaticity,
    # and stereochemical constraints are inherited from RDKit's molecular graph.
    automorphisms = ref_mol_heavy.GetSubstructMatches(
        ref_mol_heavy,
        uniquify=False,
        useChirality=True,
        maxMatches=max_automorphisms,
    )
    if not automorphisms:
        raise RuntimeError("RDKit found no graph-preserving reference atom mappings")

    best = np.inf
    for mapping in automorphisms:
        mapped_pose = pose_coords_ref_order[np.asarray(mapping, dtype=int)]
        delta = ref_coords - mapped_pose
        rmsd = float(np.sqrt(np.mean(np.sum(delta * delta, axis=1))))
        if rmsd < best:
            best = rmsd

    return best, len(automorphisms)


# ---- Load reference and establish input atom map ----
ref_mol_full, ref_mol_heavy, crystal_coords, crystal_elements = get_heavy_reference(
    CRYSTAL_LIGAND_SDF
)
input_records = parse_pdbqt_heavy_atoms(CRYSTAL_LIGAND_PDBQT)
input_elements = [r["element"] for r in input_records]
input_coords = np.asarray([r["xyz"] for r in input_records], dtype=float)

print("Reference SDF heavy atoms:", len(crystal_coords))
print("Prepared PDBQT heavy atoms:", len(input_records))
print("Reference elements:", Counter(crystal_elements))
print("Prepared PDBQT elements:", Counter(input_elements))
print("Reference centroid:", np.round(crystal_coords.mean(axis=0), 3))
print("Prepared centroid:", np.round(input_coords.mean(axis=0), 3))

# input_to_ref[k] = reference SDF heavy-atom index corresponding to input PDBQT atom k
input_to_ref, mapping_deviations, mapping_gaps = coordinate_mapping_sdf_to_pdbqt(
    crystal_coords, crystal_elements, input_records,
    tolerance_angstrom=0.01, uniqueness_margin_angstrom=0.05
)

# Invert once so each reference atom index can be populated from a pose in Vina input order.
ref_to_input = np.full(len(crystal_coords), -1, dtype=int)
for input_i, ref_i in enumerate(input_to_ref):
    ref_to_input[ref_i] = input_i
if np.any(ref_to_input < 0):
    raise RuntimeError("Could not invert the input atom mapping")

# Save an auditable atom correspondence report, including the reference graph context.
atom_map_rows = []
for input_i, ref_i in enumerate(input_to_ref):
    atom = ref_mol_heavy.GetAtomWithIdx(int(ref_i))
    neighbors = []
    for bond in atom.GetBonds():
        other = bond.GetOtherAtom(atom)
        neighbors.append(f"{other.GetIdx()}:{other.GetSymbol()}({str(bond.GetBondType())})")
    atom_map_rows.append({
        "pdbqt_atom_index_0based": int(input_i),
        "pdbqt_atom_name": input_records[input_i]["atom_name"],
        "pdbqt_atom_type": input_records[input_i]["atom_type"],
        "element": input_records[input_i]["element"],
        "reference_sdf_atom_index_0based": int(ref_i),
        "coordinate_deviation_A": float(mapping_deviations[input_i]),
        "nearest_neighbor_gap_A": (float(mapping_gaps[input_i])
                                    if np.isfinite(mapping_gaps[input_i]) else None),
        "reference_degree": int(atom.GetDegree()),
        "reference_neighbors_bonds": "; ".join(sorted(neighbors)),
    })
ANALYSIS_DIR.mkdir(parents=True, exist_ok=True)
atom_map_path = ANALYSIS_DIR / "1JI_input_atom_mapping_audit.csv"
pd.DataFrame(atom_map_rows).to_csv(atom_map_path, index=False)
print(f"Atom-mapping audit saved: {atom_map_path}")

# Validate the reference graph's self-mapping enumeration before processing seeds.
automorphisms = ref_mol_heavy.GetSubstructMatches(
    ref_mol_heavy, uniquify=False, useChirality=True, maxMatches=1000000
)
if not automorphisms:
    raise RuntimeError("No graph-preserving automorphisms found for reference ligand")
print(f"Graph-valid reference mappings considered: {len(automorphisms)}")

# ---- Validate every Vina pose and calculate symmetry-corrected direct RMSD ----
rows = []
for seed in SEEDS:
    pose_path = Path(REDOCKING_1JI_DIR) / f"1JI_redocked_seed_{seed}.pdbqt"
    pose_records = parse_pdbqt_heavy_atoms(pose_path)
    pose_coords_input_order = np.asarray([r["xyz"] for r in pose_records], dtype=float)
    pose_elements = [r["element"] for r in pose_records]

    if len(pose_records) != len(input_records):
        raise RuntimeError(
            f"Seed {seed}: heavy-atom count mismatch: input={len(input_records)}, "
            f"pose={len(pose_records)}"
        )
    if pose_elements != input_elements:
        raise RuntimeError(
            f"Seed {seed}: PDBQT heavy-atom element order differs from input ligand. "
            "Do not assume atom-index correspondence."
        )
    input_names = [r["atom_name"] for r in input_records]
    pose_names = [r["atom_name"] for r in pose_records]
    input_types = [r["atom_type"] for r in input_records]
    pose_types = [r["atom_type"] for r in pose_records]
    # Duplicate atom names are permitted: converters sometimes use generic element
    # labels. Identity is tracked by record order plus element/type sequence,
    # relying on Vina preserving the prepared ligand atom order.
    if pose_names != input_names:
        raise RuntimeError(
            f"Seed {seed}: PDBQT atom-name order differs from input ligand. "
            "Inspect atom ordering before calculating RMSD."
        )
    if pose_types != input_types:
        raise RuntimeError(
            f"Seed {seed}: AutoDock atom-type sequence differs from the prepared input. "
            "Inspect ligand preparation and pose parsing before calculating RMSD."
        )

    # Reorder the pose coordinates into the reference SDF heavy-atom index order.
    pose_coords_ref_order = np.empty_like(crystal_coords)
    for input_i, ref_i in enumerate(input_to_ref):
        pose_coords_ref_order[ref_i] = pose_coords_input_order[input_i]

    rmsd, n_maps = direct_symmetry_corrected_rmsd(
        ref_mol_heavy, crystal_coords, pose_coords_ref_order
    )
    rows.append({
        "seed": int(seed),
        "heavy_atoms": len(crystal_coords),
        "rmsd_A": rmsd,
        "within_2A": bool(rmsd <= 2.0),
        "graph_mappings_considered": int(n_maps),
        "superposition": False,
    })

rmsd_df = pd.DataFrame(rows).sort_values("seed").reset_index(drop=True)
ANALYSIS_DIR.mkdir(parents=True, exist_ok=True)
rmsd_csv = ANALYSIS_DIR / "1JI_crystal_vs_redocked_RMSD.csv"
rmsd_df.to_csv(rmsd_csv, index=False)

print("\nSymmetry-corrected direct redocking RMSD (no superposition):")
print(rmsd_df.to_string(index=False))
print(f"\nRMSD table saved: {rmsd_csv}")

best_idx = rmsd_df["rmsd_A"].idxmin()
best_seed = int(rmsd_df.loc[best_idx, "seed"])
print(f"Best seed by symmetry-corrected direct RMSD: {best_seed}")
print("Note: the 2.0 A criterion is applied to graph-mapped heavy-atom RMSD.")


Reference SDF heavy atoms: 32
Prepared PDBQT heavy atoms: 32
Reference elements: Counter({'C': 25, 'N': 4, 'O': 2, 'CL': 1})
Prepared PDBQT elements: Counter({'C': 25, 'N': 4, 'O': 2, 'CL': 1})
Reference centroid: [22.765 -2.076 -1.167]
Prepared centroid: [22.765 -2.076 -1.167]
Note: repeated generic PDBQT atom labels (not used as unique IDs): ['C', 'N', 'O']
Input SDF-to-PDBQT mapping RMSD: 0.000000 A
Input SDF-to-PDBQT maximum deviation: 0.000000 A
Minimum same-element nearest-neighbor gap: 1.362512 A
Atom-mapping audit saved: /content/drive/MyDrive/DDR1_Workflow/ddr1_redocking/analysis/1JI_input_atom_mapping_audit.csv
Graph-valid reference mappings considered: 2

Symmetry-corrected direct redocking RMSD (no superposition):
 seed  heavy_atoms   rmsd_A  within_2A  graph_mappings_considered  superposition
    7           32 0.232793       True                          2          False
   42           32 0.266834       True                          2          False
 2026           32 0.

In [10]:
# CELL 8 — Build complexes for PLIP using config.write_ligand_pdb
STRUCT_DIR = ANALYSIS_DIR / "structures"
STRUCT_DIR.mkdir(parents=True, exist_ok=True)

receptor_atoms = [L.rstrip("\n") for L in RECEPTOR_PDB.read_text().splitlines()
                  if L.startswith("ATOM")]
crystal_ligand = [L.rstrip("\n") for L in CRYSTAL_LIGAND_PDB.read_text().splitlines()
                  if L.startswith(("ATOM", "HETATM"))]
crystal_complex = STRUCT_DIR / "1JI_crystal_complex.pdb"
crystal_complex.write_text("\n".join(receptor_atoms + crystal_ligand) + "\nEND\n")

best_pose = REDOCKING_1JI_DIR / f"1JI_redocked_seed_{best_seed}.pdbqt"
ligand_block = STRUCT_DIR / f"1JI_redocked_seed_{best_seed}_ligand.pdb"
write_ligand_pdb(best_pose, ligand_block, resname="LIG", chain="Z", resnum=1001)
redocked_ligand = [L.rstrip("\n") for L in ligand_block.read_text().splitlines()
                   if L.startswith(("ATOM", "HETATM"))]
redocked_complex = STRUCT_DIR / f"1JI_redocked_complex_seed_{best_seed}.pdb"
redocked_complex.write_text("\n".join(receptor_atoms + redocked_ligand) + "\nEND\n")
print(f"Crystal complex : {crystal_complex}")
print(f"Redocked complex: {redocked_complex}")

Crystal complex : /content/drive/MyDrive/DDR1_Workflow/ddr1_redocking/analysis/structures/1JI_crystal_complex.pdb
Redocked complex: /content/drive/MyDrive/DDR1_Workflow/ddr1_redocking/analysis/structures/1JI_redocked_complex_seed_7.pdb


In [13]:
# CELL 9 — PLIP on crystal and redocked complexes, plus interaction comparison
# Depends on: crystal_complex, redocked_complex, best_seed (cell 8),
#             ANALYSIS_DIR, REPORTS_DIR, subprocess, ET (this cell).

import xml.etree.ElementTree as ET

PLIP_EXE = shutil.which("plip")
if PLIP_EXE is None:
    raise FileNotFoundError("PLIP executable not found on PATH")

PLIP_ROOT = ANALYSIS_DIR / "PLIP"
PLIP_ROOT.mkdir(parents=True, exist_ok=True)


def run_plip(complex_pdb, out_dir, name):
    """Run PLIP, return the output directory. Raise on failure."""
    out_dir.mkdir(parents=True, exist_ok=True)
    cmd = [PLIP_EXE, "-f", str(complex_pdb), "-o", str(out_dir),
           "-x", "-t", "--name", name]
    r = subprocess.run(cmd, capture_output=True, text=True)
    if r.returncode != 0:
        raise RuntimeError(
            f"PLIP failed for {name} (rc={r.returncode})\n"
            f"STDOUT:\n{r.stdout[-2000:]}\n"
            f"STDERR:\n{r.stderr[-2000:]}"
        )
    xmls = list(out_dir.glob("*.xml"))
    if not xmls:
        raise RuntimeError(f"PLIP produced no XML for {name} in {out_dir}")
    return out_dir


crystal_dir = run_plip(crystal_complex, PLIP_ROOT / "crystal", "1JI_crystal")
redocked_dir = run_plip(
    redocked_complex,
    PLIP_ROOT / f"redocked_seed_{best_seed}",
    f"1JI_redocked_seed_{best_seed}",
)

# ---------------------------------------------------------------
# 1. PLIP XML parsing
# ---------------------------------------------------------------
TAG_MAP = {
    "hydrophobic_interaction": "hydrophobic",
    "hydrogen_bond": "hbond",
    "water_bridge": "water_bridge",
    "salt_bridge": "salt_bridge",
    "pi_stack": "pi_stack",
    "pistacking": "pi_stack",
    "pi_cation_interaction": "pi_cation",
    "pi_cation": "pi_cation",
    "pication": "pi_cation",
    "halogen_bond": "halogen_bond",
    "metal_complex": "metal_complex",
}

# Fields PLIP uses to describe residues. Different builds have used
# different names, so we collect all of them and pick whichever exists.
RESIDUE_NUMBER_KEYS = ("resnr", "resnum", "residue_number", "prot_idx", "protnr")
RESIDUE_NAME_KEYS = ("restype", "resname", "residue_name", "prot_type")


def local(tag):
    return tag.split("}")[-1]


def _pick(el, names):
    """Return the first non-empty text value from any descendant tag in `names`."""
    wanted = {n.lower() for n in names}
    for child in el.iter():
        if local(child.tag).lower() in wanted:
            txt = (child.text or "").strip()
            if txt:
                return txt
    return ""


def parse_plip_xml(path, label):
    """Return a DataFrame of interaction records from a PLIP XML file."""
    root = ET.parse(path).getroot()
    rows = []
    for el in root.iter():
        tag = local(el.tag)
        if tag not in TAG_MAP:
            continue
        rec = {"pose": label, "type": TAG_MAP[tag]}
        # Capture all direct + nested text fields so we can inspect them later
        for child in el.iter():
            if child is el:
                continue
            ctag = local(child.tag)
            txt = (child.text or "").strip()
            if txt:
                rec[ctag] = txt
        # Canonical residue fields
        rec["resnr"] = _pick(el, RESIDUE_NUMBER_KEYS)
        rec["restype"] = _pick(el, RESIDUE_NAME_KEYS)
        rows.append(rec)
    return pd.DataFrame(rows)


crystal_xml = next(crystal_dir.glob("*.xml"))
redocked_xml = next(redocked_dir.glob("*.xml"))
crystal_df = parse_plip_xml(crystal_xml, "crystal")
redocked_df = parse_plip_xml(redocked_xml, f"seed_{best_seed}")

# ---------------------------------------------------------------
# 2. Residue-field sanity check
# ---------------------------------------------------------------
for label, df in (("crystal", crystal_df), (f"seed_{best_seed}", redocked_df)):
    if df.empty:
        print(f"⚠ {label}: no interaction records in XML")
        continue
    n_with_residue = int(
        ((df["resnr"].astype(str).str.strip() != "")
         & (df["restype"].astype(str).str.strip() != "")).sum()
    )
    if n_with_residue == 0:
        print(f"⚠ {label}: XML has {len(df)} records but none carry a residue "
              f"number + name. The residue-level comparison below will be empty. "
              f"Check the PLIP build's XML schema.")

# ---------------------------------------------------------------
# 3. Interaction-type counts (context only)
# ---------------------------------------------------------------
def summarize(df):
    if df.empty:
        return pd.DataFrame(columns=["type", "count"])
    return (df.groupby("type").size()
              .reset_index(name="count")
              .sort_values("type")
              .reset_index(drop=True))


c = summarize(crystal_df).rename(columns={"count": "crystal"})
r = summarize(redocked_df).rename(columns={"count": "redocked"})
cmp = c.merge(r, on="type", how="outer").fillna(0)
for col in ("crystal", "redocked"):
    cmp[col] = cmp[col].astype(int)
cmp["delta"] = cmp["redocked"] - cmp["crystal"]
cmp.to_csv(REPORTS_DIR / "1JI_PLIP_interaction_comparison.csv", index=False)

print("\nInteraction-type counts (crystal vs best redocked pose):")
print(cmp.to_string(index=False))

# ---------------------------------------------------------------
# 4. Residue-level comparison (the important one)
# ---------------------------------------------------------------
def residue_set(df, itype):
    """Set of 'RESTYPE RESNR' strings for one interaction type.

    Handles the case where PLIP leaves a residue field blank, or where
    pandas stores it as NaN.
    """
    if df.empty or "type" not in df.columns:
        return set()
    sub = df[df["type"] == itype]
    out = set()
    for _, row in sub.iterrows():
        resnr = row.get("resnr", None)
        restype = row.get("restype", None)
        if pd.isna(resnr) or pd.isna(restype):
            continue
        rn = str(resnr).strip()
        rt = str(restype).strip()
        if rn and rt and rn.lower() != "nan" and rt.lower() != "nan":
            out.add(f"{rt}{rn}")
    return out


ITYPES = ["hbond", "hydrophobic", "salt_bridge", "pi_stack",
          "pi_cation", "halogen_bond", "water_bridge"]

residue_rows = []
for itype in ITYPES:
    crystal_res = residue_set(crystal_df, itype)
    redocked_res = residue_set(redocked_df, itype)
    matched = crystal_res & redocked_res
    missed = crystal_res - redocked_res
    new = redocked_res - crystal_res
    residue_rows.append({
        "interaction": itype,
        "crystal_residues": ";".join(sorted(crystal_res)),
        "redocked_residues": ";".join(sorted(redocked_res)),
        "matched": ";".join(sorted(matched)),
        "missed_from_crystal": ";".join(sorted(missed)),
        "new_in_redocked": ";".join(sorted(new)),
        "n_crystal": len(crystal_res),
        "n_redocked": len(redocked_res),
        "n_matched": len(matched),
        "n_missed": len(missed),
        "n_new": len(new),
    })

residue_df = pd.DataFrame(residue_rows)
residue_df.to_csv(REPORTS_DIR / "1JI_PLIP_residue_comparison.csv", index=False)

print("\nResidue-level PLIP comparison:")
print(residue_df[["interaction", "n_crystal", "n_matched",
                  "n_missed", "n_new"]].to_string(index=False))

# ---------------------------------------------------------------
# 5. Pass criterion: every crystal H-bond residue is reproduced
# ---------------------------------------------------------------
hb_pass = None
hb_row = residue_df[residue_df["interaction"] == "hbond"]
if hb_row.empty or int(hb_row.iloc[0]["n_crystal"]) == 0:
    print("\n⚠ No crystallographic H-bond residues were identified in the XML. "
          "Skipping the H-bond pass criterion.")
else:
    n_missed_hb = int(hb_row.iloc[0]["n_missed"])
    n_crystal_hb = int(hb_row.iloc[0]["n_crystal"])
    if n_missed_hb == 0:
        hb_pass = True
        print(f"\n✓ All {n_crystal_hb} crystallographic H-bond residues are "
              f"present in the redocked pose.")
    else:
        hb_pass = False
        missed = hb_row.iloc[0]["missed_from_crystal"]
        print(f"\n⚠ Redocked pose misses {n_missed_hb} of {n_crystal_hb} "
              f"crystal H-bond residue(s): {missed}")

# ---------------------------------------------------------------
# 6. Compact summary of the crystal hydrogen-bond set
# ---------------------------------------------------------------
crystal_hb = sorted(residue_set(crystal_df, "hbond"))
redocked_hb = sorted(residue_set(redocked_df, "hbond"))
print(f"\nCrystal H-bond residues  : {', '.join(crystal_hb) or '(none)'}")
print(f"Redocked H-bond residues : {', '.join(redocked_hb) or '(none)'}")


Interaction-type counts (crystal vs best redocked pose):
       type  crystal  redocked  delta
      hbond        4         4      0
hydrophobic        8         9      1

Residue-level PLIP comparison:
 interaction  n_crystal  n_matched  n_missed  n_new
       hbond          3          3         0      0
 hydrophobic          8          7         1      1
 salt_bridge          0          0         0      0
    pi_stack          0          0         0      0
   pi_cation          0          0         0      0
halogen_bond          0          0         0      0
water_bridge          0          0         0      0

✓ All 3 crystallographic H-bond residues are present in the redocked pose.

Crystal H-bond residues  : ASP778, GLU672, MET704
Redocked H-bond residues : ASP778, GLU672, MET704


In [14]:
# CELL 10 — Summary JSON
summary = {
    "receptor_pdbqt": str(RECEPTOR_PDBQT),
    "receptor_sha256": sha256_file(RECEPTOR_PDBQT),
    "ligand_pdbqt": str(CRYSTAL_LIGAND_PDBQT),
    "ligand_sha256": sha256_file(CRYSTAL_LIGAND_PDBQT),
    "box": BOX,
    "seeds": SEEDS,
    "best_seed": best_seed,
    "rmsd_table": rmsd_df.to_dict(orient="records"),
    "interaction_comparison": cmp.to_dict(orient="records"),
    "residue_comparison": residue_df.to_dict(orient="records"),
    "hbond_pass_criterion_met": hb_pass,
    "rmsd_method": "Graph-mapped symmetry-corrected direct heavy-atom RMSD; no superposition.",
    "input_coordinate_mapping_qc": "Passed strict SDF-to-PDBQT coordinate-preservation QC before RMSD calculation.",
    "note": "Redocking validation tests reproduction of the crystallographic pose; it does not by itself validate predictions for other ligands.",
}
with open(REPORTS_DIR / "1JI_redocking_summary.json", "w") as f:
    json.dump(summary, f, indent=2, default=str)

print("All seeds within 2 A:", bool(rmsd_df["within_2A"].all()))
print(f"H-bond residue reproduction: {hb_pass}")
print("Next: run 3-DDR1-DOCK.")

All seeds within 2 A: True
H-bond residue reproduction: True
Next: run 3-DDR1-DOCK.
